# 02 - Silver Transform

Limpieza, tipado, reglas de calidad y enriquecimiento de datos para análisis fintech.

Transformaciones principales:
- Tipado de fechas y montos.
- Normalización de texto.
- Eliminación de duplicados.
- Join entre transacciones, clientes y alertas.
- Cálculo de banderas de riesgo.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog_name", "fintech_lakehouse")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")

catalog_name = dbutils.widgets.get("catalog_name")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")

spark.sql(f"USE CATALOG {catalog_name}")

In [ ]:
customers = spark.table(f"{catalog_name}.{bronze_schema}.customers_bronze")
transactions = spark.table(f"{catalog_name}.{bronze_schema}.transactions_bronze")
alerts = spark.table(f"{catalog_name}.{bronze_schema}.fraud_alerts_bronze")

customers_silver = (
    customers
    .dropDuplicates(["customer_id"])
    .withColumn("signup_date", F.to_date("signup_date"))
    .withColumn("segment", F.lower(F.trim("segment")))
    .withColumn("district", F.initcap(F.trim("district")))
    .withColumn("kyc_status", F.lower(F.trim("kyc_status")))
    .withColumn("risk_level", F.lower(F.trim("risk_level")))
    .withColumn("age", F.col("age").cast("int"))
    .withColumn("monthly_income_pen", F.col("monthly_income_pen").cast("decimal(18,2)"))
    .filter(F.col("customer_id").isNotNull())
)

transactions_silver = (
    transactions
    .dropDuplicates(["transaction_id"])
    .withColumn("transaction_ts", F.to_timestamp("transaction_ts"))
    .withColumn("transaction_date", F.to_date("transaction_ts"))
    .withColumn("channel", F.lower(F.trim("channel")))
    .withColumn("transaction_type", F.lower(F.trim("transaction_type")))
    .withColumn("merchant_category", F.lower(F.trim("merchant_category")))
    .withColumn("status", F.lower(F.trim("status")))
    .withColumn("amount_pen", F.col("amount_pen").cast("decimal(18,2)"))
    .withColumn("is_cross_border", F.col("is_cross_border").cast("int"))
    .withColumn("is_fraud_confirmed", F.col("is_fraud_confirmed").cast("int"))
    .filter((F.col("transaction_id").isNotNull()) & (F.col("customer_id").isNotNull()) & (F.col("amount_pen") >= 0))
)

alerts_silver = (
    alerts
    .dropDuplicates(["alert_id"])
    .withColumn("score", F.col("score").cast("double"))
    .withColumn("created_ts", F.to_timestamp("created_ts"))
    .withColumn("alert_rule", F.lower(F.trim("alert_rule")))
    .withColumn("alert_status", F.lower(F.trim("alert_status")))
    .filter((F.col("alert_id").isNotNull()) & (F.col("transaction_id").isNotNull()))
)

In [ ]:
tx_customer = (
    transactions_silver.alias("tx")
    .join(customers_silver.alias("cu"), on="customer_id", how="left")
)

alert_by_tx = (
    alerts_silver
    .groupBy("transaction_id")
    .agg(
        F.count("*").alias("alert_count"),
        F.max("score").alias("max_alert_score"),
        F.first("alert_rule", ignorenulls=True).alias("main_alert_rule"),
        F.max(F.when(F.col("alert_status") == "escalated", 1).otherwise(0)).alias("has_escalated_alert")
    )
)

transactions_enriched_silver = (
    tx_customer
    .join(alert_by_tx, on="transaction_id", how="left")
    .fillna({"alert_count": 0, "max_alert_score": 0.0, "has_escalated_alert": 0})
    .withColumn(
        "risk_score",
        F.least(
            F.lit(100.0),
            F.col("max_alert_score")
            + F.when(F.col("risk_level") == "high", 20).otherwise(0)
            + F.when(F.col("amount_pen") > 700, 10).otherwise(0)
            + F.when(F.col("is_cross_border") == 1, 10).otherwise(0)
        )
    )
    .withColumn(
        "risk_bucket",
        F.when(F.col("risk_score") >= 75, "high")
         .when(F.col("risk_score") >= 45, "medium")
         .otherwise("low")
    )
)

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{silver_schema}")

customers_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog_name}.{silver_schema}.customers_silver")
transactions_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog_name}.{silver_schema}.transactions_silver")
alerts_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog_name}.{silver_schema}.fraud_alerts_silver")
transactions_enriched_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog_name}.{silver_schema}.transactions_enriched_silver")

display(transactions_enriched_silver.limit(20))